In [1]:
import requests
import json
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import random

In [81]:
# GETS SEASON TRANSACTIONS FROM STANLEY CAP

dates= []
teams = []
players = []
transaction_details = []

url = "https://www.thestanleycap.com/transactions/view_transactions/20242025"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")
transaction_table = soup.find('table', id='transaction_table')
for row in transaction_table.find_all('tr')[1:]:  # Skip header
    cells = row.find_all('td')
    if len(cells) >= 4:
        date = cells[0].text.strip()
        team = cells[1].text.strip()
        player = cells[2].text.strip()
        details = cells[3].text.strip()
        dates.append(date)
        teams.append(team)
        players.append(player)
        transaction_details.append(details)

# Create a DataFrame from the lists
df = pd.DataFrame({
    'Date': dates,
    'Team': teams,
    'Player': players,
    'Transaction Details': transaction_details
})
df.head()
df.to_csv('transactions.csv', index=False)
    


In [83]:
# GET PLAYER CONTRACTS FROM STANLEY CAP

players = []
teams = []
positions = []
signedDates = []
startSeasons = []
endSeasons = []
years = []
totalValues = []
capHits = []
signBonuses = []
baseSalaries = []
perfBonuses = []
contractTerms = []

url = "https://www.thestanleycap.com/transactions/active_contracts/20242025"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")
transaction_table = soup.find('table', id='transaction_table')
for row in transaction_table.find_all('tr')[1:]:  # Skip header
    cells = row.find_all('td')
    if len(cells) >= 4:
        player = cells[0].text.strip()
        position = cells[1].text.strip()
        team = cells[2].text.strip()
        signedDate = cells[3].text.strip()
        startSeason = cells[4].text.strip()
        endSeason = cells[5].text.strip()
        length = cells[6].text.strip()
        totalValue = cells[7].text.strip()
        capHit = cells[8].text.strip()
        signBonus = cells[9].text.strip()
        baseSalary = cells[10].text.strip()
        perfBonus = cells[11].text.strip()
        terms = cells[12].text.strip()

        players.append(player)
        teams.append(team)
        positions.append(position)
        signedDates.append(signedDate)
        startSeasons.append(startSeason)
        endSeasons.append(endSeason)
        years.append(length)
        totalValues.append(totalValue)
        capHits.append(capHit)
        signBonuses.append(signBonus)
        baseSalaries.append(baseSalary)
        perfBonuses.append(perfBonus)
        contractTerms.append(terms)


# Create a DataFrame from the lists
df = pd.DataFrame({
    'Player': players,
    'Team': teams,
    'Position': positions,
    'Signed Date': signedDates,
    'Start Season': startSeasons,
    'End Season': endSeasons,
    'Years': years,
    'Total Value': totalValues,
    'Cap Hit': capHits,
    'Signing Bonus': signBonuses,
    'Base Salary': baseSalaries,
    'Performance Bonus': perfBonuses,
    'Terms': terms
})
df.head()

df.to_csv('stanleycap_contracts.csv', index=False)



In [ ]:
contract_table = soup.find('table', id='active_contracts')

None


In [18]:
url = "https://api.nhle.com/stats/rest/en/team"
response = requests.get(url)
team_data = response.json()
tricodes = []
for team in team_data['data']:
    tricodes.append(team['triCode'])


base_url = "https://api-web.nhle.com/v1/prospects/"
prospect_id = []
tricodes_df = []

for tricode in tricodes:
    url = base_url + tricode
    data = requests.get(url).json()
    forward_prospects = data['forwards']
    defense_prospects = data['defensemen']
    goalie_prospects = data['goalies']

    for prospect in forward_prospects:
        prospect_id.append(prospect['id'])
        tricodes_df.append(tricode)
    for prospect in defense_prospects:
        prospect_id.append(prospect['id'])
        tricodes_df.append(tricode)
    for prospect in goalie_prospects:
        prospect_id.append(prospect['id'])
        tricodes_df.append(tricode)

prospect_df = pd.DataFrame({
    'prospect_id': prospect_id,
    'tricode': tricodes_df
})
prospect_df.to_csv('prospect_ids.csv', index=False)
print(prospect_df.head())

   prospect_id tricode
0      8483424     MTL
1      8483549     MTL
2      8483728     MTL
3      8484448     MTL
4      8482081     MTL


Season Overview Lookback Stats (skater and goalie leaders)

In [ ]:
seasonID = []; playerID = []; seasonType = []; statCategory = []; statValue = []

seasonList = ['19171918', '19181919', '19191920', '19201921', '19211922', '19221923', '19231924', '19241925', '19251926', '19261927', '19271928', '19281929', '19291930', '19301931', '19311932', '19321933', '19331934', '19341935', '19351936', '19361937', '19371938', '19381939', '19391940', '19401941', '19411942', '19421943', '19431944', '19441945', '19451946']
seasonList += ['19461947', '19471948', '19481949', '19491950', '19501951', '19511952', '19521953', '19531954', '19541955', '19551956', '19561957', '19571958', '19581959', '19591960', '19601961', '19611962', '19621963', '19631964', '19641965', '19651966', '19661967', '19671968', '19681969', '19691970', '19701971', '19711972', '19721973', '19731974', '19741975']
seasonList += ['19751976', '19761977', '19771978', '19781979', '19791980', '19801981', '19811982', '19821983', '19831984', '19841985', '19851986', '19861987', '19871988', '19881989', '19891990', '19901991', '19911992', '19921993', '19931994', '19941995', '19951996', '19961997', '19971998', '19981999', '19992000']
seasonList += ['20002001', '20012002', '20022003', '20032004', '20042005', '20052006', '20062007', '20072008', '20082009', '20092010', '20102011', '20112012', '20122013', '20132014', '20142015', '20152016', '20162017', '20172018', '20182019', '20192020', '20202021', '20212022', '20222023', '20232024']

seasonTypeList = ['2', '3']

# dont include 20242025 season because it is not over yet, deal with this separately

base_url = 'https://api-web.nhle.com/v1/skater-stats-leaders/'
for season in seasonList:
    for type in seasonTypeList:
        url = base_url + season + '/' + type
        # url = 'https://api-web.nhle.com/v1/skater-stats-leaders/20232024/3'
        response = requests.get(url)
        if response.status_code == 404:
            print("Page not found (404).", playerID)
            continue
        else:
            data = response.json()
            # print(data.keys())
            for category in data.keys():
                for player in data[category]:
                    seasonID.append(season)
                    playerID.append(player['id'])
                    seasonType.append(type)
                    statCategory.append(category)
                    statValue.append(player['value'])
            # for player in data['data']:
        
        #     print(player)

# Create a DataFrame from the lists
df = pd.DataFrame({
    'seasonID': seasonID,
    'playerID': playerID,
    'seasonType': seasonType,
    'statCategory': statCategory,
    'statValue': statValue
})
df.head()
df.to_csv('skater-past-season-leaders.csv', index=False)


In [ ]:
seasonID = []; playerID = []; seasonType = []; statCategory = []; statValue = []

seasonList = ['19171918', '19181919', '19191920', '19201921', '19211922', '19221923', '19231924', '19241925', '19251926', '19261927', '19271928', '19281929', '19291930', '19301931', '19311932', '19321933', '19331934', '19341935', '19351936', '19361937', '19371938', '19381939', '19391940', '19401941', '19411942', '19421943', '19431944', '19441945', '19451946']
seasonList += ['19461947', '19471948', '19481949', '19491950', '19501951', '19511952', '19521953', '19531954', '19541955', '19551956', '19561957', '19571958', '19581959', '19591960', '19601961', '19611962', '19621963', '19631964', '19641965', '19651966', '19661967', '19671968', '19681969', '19691970', '19701971', '19711972', '19721973', '19731974', '19741975']
seasonList += ['19751976', '19761977', '19771978', '19781979', '19791980', '19801981', '19811982', '19821983', '19831984', '19841985', '19851986', '19861987', '19871988', '19881989', '19891990', '19901991', '19911992', '19921993', '19931994', '19941995', '19951996', '19961997', '19971998', '19981999', '19992000']
seasonList += ['20002001', '20012002', '20022003', '20032004', '20042005', '20052006', '20062007', '20072008', '20082009', '20092010', '20102011', '20112012', '20122013', '20132014', '20142015', '20152016', '20162017', '20172018', '20182019', '20192020', '20202021', '20212022', '20222023', '20232024']

seasonTypeList = ['2', '3']

# dont include 20242025 season because it is not over yet, deal with this separately

base_url = 'https://api-web.nhle.com/v1/goalie-stats-leaders/'
for season in seasonList:
    for type in seasonTypeList:
        url = base_url + season + '/' + type
        response = requests.get(url)
        if response.status_code == 404:
            print("Page not found (404).", playerID)
            continue
        else:
            data = response.json()
            # print(data.keys())
            for category in data.keys():
                for player in data[category]:
                    seasonID.append(season)
                    playerID.append(player['id'])
                    seasonType.append(type)
                    statCategory.append(category)
                    statValue.append(player['value'])
        # for player in data['data']:
        #     print(player)

# Create a DataFrame from the lists
df = pd.DataFrame({
    'seasonID': seasonID,
    'playerID': playerID,
    'seasonType': seasonType,
    'statCategory': statCategory,
    'statValue': statValue
})
df.head()
df.to_csv('goalie-past-season-leaders.csv', index=False)


JSONDecodeError: Expecting value: line 1 column 1 (char 0)